# Feature Engineering — Retail Customer Segmentation

Cleans the raw 3,900-customer dataset and engineers the behavioral features
(promo-dependency, value score/tier, satisfaction, two loyalty definitions,
estimated annual revenue, customer segment) consumed by the SQL / dbt layer,
Power BI, and Tableau. Produces `data/processed/customers_enriched.csv`.

Run from the repo root or from `notebooks/` — paths resolve either way.

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolve the repo root so the notebook runs from a clean clone,
# whether executed from the repo root or from the notebooks/ folder.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists() and (BASE_DIR.parent / "data").exists():
    BASE_DIR = BASE_DIR.parent

In [2]:
# ─────────────────────────────
# 1A  LOAD & CLEAN RAW DATA
# ─────────────────────────────
print("=" * 65)
print("STEP 1A — Loading & Cleaning Raw Dataset")
print("=" * 65)

RAW_PATH = BASE_DIR / "data" / "raw" / "Dataset.csv"
OUT_DIR  = BASE_DIR / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f"  Rows loaded : {len(df):,}")
print(f"  Columns     : {df.columns.tolist()}")

STEP 1A — Loading & Cleaning Raw Dataset
  Rows loaded : 3,900
  Columns     : ['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']


In [3]:
# ── Rename for SQL-friendliness ────────────────────────────────
df.columns = [
    "customer_id", "age", "gender", "item_purchased", "category",
    "purchase_amount_usd", "location", "size", "color", "season",
    "review_rating", "subscription_status", "shipping_type",
    "discount_applied", "promo_code_used", "previous_purchases",
    "payment_method", "frequency_of_purchases"
]

In [4]:
df.isnull().sum()

customer_id                0
age                        0
gender                     0
item_purchased             0
category                   0
purchase_amount_usd        0
location                   0
size                       0
color                      0
season                     0
review_rating             37
subscription_status        0
shipping_type              0
discount_applied           0
promo_code_used            0
previous_purchases         0
payment_method             0
frequency_of_purchases     0
dtype: int64

In [5]:
# ── Handle the only 37 nulls — Review Rating ────────────
# Capture the audit flag BEFORE imputing — otherwise it is always 0.
median_rating = df["review_rating"].median()
df["rating_was_imputed"] = df["review_rating"].isna().astype(int)
df["review_rating"] = df["review_rating"].fillna(median_rating)
print(f"  Imputed {int(df['rating_was_imputed'].sum())} null ratings "
      f"with median = {median_rating}")
print(f"  Nulls after imputation: {df.isnull().sum().sum()}")

  Imputed 37 null ratings with median = 3.8
  Nulls after imputation: 0


In [6]:
# ── Binary encode Yes/No columns ──────────────────────────────
bool_cols = ["subscription_status", "discount_applied", "promo_code_used"]
for col in bool_cols:
    df[col + "_flag"] = (df[col].str.strip().str.lower() == "yes").astype(int)

print("  Boolean flags created for:", bool_cols)

  Boolean flags created for: ['subscription_status', 'discount_applied', 'promo_code_used']


In [7]:
# ── Purchase frequency → numeric purchase_per_year ────────────
freq_map = {
    "weekly":         52,
    "bi-weekly":      26,
    "fortnightly":    26,    # same as bi-weekly
    "monthly":        12,
    "quarterly":       4,
    "every 3 months":  4,
    "annually":        1,
}
df["freq_lower"] = df["frequency_of_purchases"].str.lower().str.strip()
df["purchase_frequency_per_year"] = df["freq_lower"].map(freq_map).fillna(12)

print(f"  Frequency mapping applied. Unique per-year values: {sorted(df['purchase_frequency_per_year'].unique())}")

  Frequency mapping applied. Unique per-year values: [np.int64(1), np.int64(4), np.int64(12), np.int64(26), np.int64(52)]


In [8]:
# ── Age bands ─────────────────────────────────────────────────
bins   = [0, 24, 34, 44, 54, 64, 120]
labels = ["Gen Z (18-24)", "Millennial-Early (25-34)",
          "Millennial-Late (35-44)", "Gen X (45-54)",
          "Boomer-Early (55-64)", "Boomer+ (65+)"]
df["age_band"] = pd.cut(df["age"], bins=bins, labels=labels, right=True)
print(f"  Age bands created: {df['age_band'].value_counts().to_dict()}")


  Age bands created: {'Millennial-Early (25-34)': 755, 'Gen X (45-54)': 752, 'Boomer-Early (55-64)': 751, 'Millennial-Late (35-44)': 729, 'Gen Z (18-24)': 486, 'Boomer+ (65+)': 427}


In [9]:
# ──────────────────────────────────────────────────────────────
# FEATURE 1: PROMOTIONAL DEPENDENCY SCORE (0–100)
# Logic:
#   A customer who ALWAYS needs a discount/promo to buy is a
#   "bargain hunter".  We capture this by combining:
#     • discount_applied_flag  (did this transaction use discount?)
#     • promo_code_used_flag   (was a promo code present?)
#   Both are binary → combined into a 0–2 raw score, scaled to 0–100.
#   Higher = more promo-dependent.
# Business value:
#   Segments customers who will churn if discounts stop vs those
#   who buy regardless — directly answers Key Question 1 & 4.
# ──────────────────────────────────────────────────────────────
df["promo_dependency_score"] = ((df["discount_applied_flag"] + df["promo_code_used_flag"]) / 2 * 100).round(1)
# print(f"  [F1] promo_dependency_score | mean={df['promo_dependency_score'].mean():.1f} | "
#       f"min={df['promo_dependency_score'].min()} | max={df['promo_dependency_score'].max()}")

In [10]:
# ──────────────────────────────────────────────────────────────
# FEATURE 2: COMPOSITE VALUE SCORE (0–100)
# Logic:
#   True customer value must combine:
#     • purchase_amount_usd  — how much they spend per visit
#     • previous_purchases   — how many times they've bought (loyalty depth)
#     • purchase_frequency_per_year — how often they come back
#   Each component is min-max normalised to 0–1, then weighted:
#     spend: 35%  |  tenure (prev purchases): 40%  |  frequency: 25%
#   Weighting rationale: tenure is weighted highest because it is
#   the hardest to fake and correlates most with LTV.
# Business value:
#   Powers the Customer Pyramid (who are the top 20% driving 80% of revenue?)
# ──────────────────────────────────────────────────────────────
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df["value_score"] = (
    0.35 * minmax(df["purchase_amount_usd"]) +
    0.40 * minmax(df["previous_purchases"]) +
    0.25 * minmax(df["purchase_frequency_per_year"])
) * 100
df["value_score"] = df["value_score"].round(2)
print(f"  [F2] value_score           | mean={df['value_score'].mean():.2f} | "
      f"p25={df['value_score'].quantile(0.25):.2f} | p75={df['value_score'].quantile(0.75):.2f}")


  [F2] value_score           | mean=45.35 | p25=32.97 | p75=57.78


In [11]:
# ──────────────────────────────────────────────────────────────
# FEATURE 3: VALUE TIER (Categorical)
# Logic:
#   Bucket value_score into 4 tiers using quartiles:
#     Q4 (top 25%)  → "Champions"
#     Q3            → "Loyalists"
#     Q2            → "Potential"
#     Q1 (bot 25%)  → "At-Risk"
# Business value:
#   Enables targeted interventions. Champions get VIP treatment.
#   At-Risk get win-back campaigns. Potential get nurturing.
# ──────────────────────────────────────────────────────────────
q25, q50, q75 = (df["value_score"].quantile([0.25, 0.50, 0.75])
                  .values)
def assign_tier(v):
    if v >= q75: return "Champions"
    elif v >= q50: return "Loyalists"
    elif v >= q25: return "Potential"
    else: return "At-Risk"

df["value_tier"] = df["value_score"].apply(assign_tier)
print(f"  [F3] value_tier distribution:")
print("      " + str(df["value_tier"].value_counts().to_dict()))

  [F3] value_tier distribution:
      {'Loyalists': 976, 'At-Risk': 975, 'Champions': 975, 'Potential': 974}


In [12]:
# ──────────────────────────────────────────────────────────────
# FEATURE 4: SATISFACTION FLAG (Binary)
# Logic:
#   review_rating ≥ 4.0 → satisfied = 1, else 0.
#   Threshold 4.0 chosen because:
#     • ratings range 2.5–5.0; median ~3.7
#     • 4.0+ represents genuine positive sentiment (top ~40%)
#     • aligns with standard e-commerce NPS-equivalent threshold
# Business value:
#   Cross-tabs satisfaction vs promo dependency to see if
#   promotions create happy customers or just transactional ones.
# ──────────────────────────────────────────────────────────────
df["satisfaction_flag"] = (df["review_rating"] >= 4.0).astype(int)
pct = df["satisfaction_flag"].mean() * 100
print(f"  [F4] satisfaction_flag     | {pct:.1f}% of customers are satisfied (rating ≥ 4.0)")

  [F4] satisfaction_flag     | 41.9% of customers are satisfied (rating ≥ 4.0)


In [13]:
# ──────────────────────────────────────────────────────────────
# FEATURE 5: LOYALTY DEFINITION A — "Behavioral Loyalty"
# Logic:
#   A customer is behaviorally loyal if they:
#     • Have previous_purchases ≥ median(previous_purchases)  [tenure depth]
#     • AND purchase_frequency_per_year ≥ 12 (buy at least monthly)
#     • AND promo_dependency_score < 50 (don't solely rely on promos)
#   This is a conservative, revenue-safe definition.
# ──────────────────────────────────────────────────────────────
med_prev = df["previous_purchases"].median()
df["is_loyal_behavioral"] = (
    (df["previous_purchases"] >= med_prev) &
    (df["purchase_frequency_per_year"] >= 12) &
    (df["promo_dependency_score"] < 50)
).astype(int)
print(f"\n  [F5-A] Behavioral Loyalty  | {df['is_loyal_behavioral'].sum():,} customers "
      f"({df['is_loyal_behavioral'].mean()*100:.1f}%)")


  [F5-A] Behavioral Loyalty  | 627 customers (16.1%)


In [14]:
# ──────────────────────────────────────────────────────────────
# FEATURE 6: LOYALTY DEFINITION B — "Value-Satisfaction Loyalty"
# Logic:
#   A customer is valuable-loyal if they:
#     • Are in Champions or Loyalists tier (value_score ≥ Q2)
#     • AND satisfaction_flag = 1 (genuinely happy customer)
#   This definition focuses on quality over quantity — fewer but
#   more commercially meaningful customers.
# Competition between definitions:
#   Def-A captures frequency-driven loyalty (great for subscription push)
#   Def-B captures value-satisfaction loyalty (great for premium pricing)
#   We argue for Def-B as the primary because satisfaction correlates
#   with organic referrals and retention WITHOUT discount dependency.
# ──────────────────────────────────────────────────────────────
df["is_loyal_value_sat"] = (
    (df["value_tier"].isin(["Champions", "Loyalists"])) &
    (df["satisfaction_flag"] == 1)
).astype(int)
print(f"  [F5-B] Value-Sat Loyalty   | {df['is_loyal_value_sat'].sum():,} customers "
      f"({df['is_loyal_value_sat'].mean()*100:.1f}%)")


  [F5-B] Value-Sat Loyalty   | 830 customers (21.3%)


In [15]:
# ── Agreement between definitions ─────────────────────────────
agree = (df["is_loyal_behavioral"] == df["is_loyal_value_sat"]).mean() * 100
print(f"  Agreement between A & B    | {agree:.1f}%")
print(f"  → Adopting Def-B (value-satisfaction) as PRIMARY loyalty label.")
print(f"    Justification: higher correlation with purchase_amount_usd and")
print(f"    satisfaction_flag, making it a stronger proxy for LTV.")


  Agreement between A & B    | 75.6%
  → Adopting Def-B (value-satisfaction) as PRIMARY loyalty label.
    Justification: higher correlation with purchase_amount_usd and
    satisfaction_flag, making it a stronger proxy for LTV.


In [16]:
# ──────────────────────────────────────────────────────────────
# FEATURE 7: ESTIMATED ANNUAL REVENUE PER CUSTOMER
# Logic:
#   purchase_amount_usd × purchase_frequency_per_year
#   This is the single most important commercial metric.
# ──────────────────────────────────────────────────────────────
df["estimated_annual_revenue"] = (
    df["purchase_amount_usd"] * df["purchase_frequency_per_year"]
).round(2)
print(f"\n  [F7] est_annual_revenue    | mean=${df['estimated_annual_revenue'].mean():,.0f} | "
      f"max=${df['estimated_annual_revenue'].max():,.0f} | "
      f"total=${df['estimated_annual_revenue'].sum():,.0f}")


  [F7] est_annual_revenue    | mean=$1,039 | max=$5,200 | total=$4,051,829


In [17]:
# ──────────────────────────────────────────────────────────────
# FEATURE 8: SEGMENT LABEL (human-readable for dashboard)
# Combines value_tier + promo_dependency for a 2-axis label
# ──────────────────────────────────────────────────────────────
def segment_label(row):
    tier = row["value_tier"]
    promo = row["promo_dependency_score"]
    if tier == "Champions" and promo < 50:
        return "Organic Champions"
    elif tier == "Champions" and promo >= 50:
        return "Promo Champions"
    elif tier == "Loyalists" and promo < 50:
        return "Stable Loyalists"
    elif tier == "Loyalists" and promo >= 50:
        return "Promo Loyalists"
    elif tier == "Potential":
        return "Growth Potential"
    else:
        return "At-Risk / Dormant"

df["customer_segment"] = df.apply(segment_label, axis=1)
print(f"  [F8] customer_segment distribution:")
for seg, cnt in df["customer_segment"].value_counts().items():
    print(f"      {seg:<30} → {cnt:4d} customers ({cnt/len(df)*100:.1f}%)")

  [F8] customer_segment distribution:
      At-Risk / Dormant              →  975 customers (25.0%)
      Growth Potential               →  974 customers (25.0%)
      Stable Loyalists               →  570 customers (14.6%)
      Organic Champions              →  539 customers (13.8%)
      Promo Champions                →  436 customers (11.2%)
      Promo Loyalists                →  406 customers (10.4%)


In [18]:
# ─────────────────────────────
# 1D  EXPORT ENRICHED DATASET
# ─────────────────────────────
print("\n" + "=" * 65)
print("STEP 1D — Exporting Enriched Dataset")
print("=" * 65)

# Drop helper columns that were only needed for engineering
df_out = df.drop(columns=["freq_lower", "rating_was_imputed"], errors="ignore")

OUT_CSV = OUT_DIR / "customers_enriched.csv"
df_out.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV}")
print(f"  Shape: {df_out.shape[0]:,} rows × {df_out.shape[1]} columns")

# ── Summary stats table ───────────────────────
print("\n── ENGINEERED FEATURE SUMMARY ───────────────────")
summary = df_out[["promo_dependency_score", "value_score",
                   "satisfaction_flag", "estimated_annual_revenue",
                   "is_loyal_behavioral", "is_loyal_value_sat"]].describe().round(2)
print(summary.to_string())

# ── Value tier revenue share ────────────────────
print("\n── REVENUE SHARE BY VALUE TIER ─────────────────")
rev_share = (df_out.groupby("value_tier")["estimated_annual_revenue"]
             .sum()
             .sort_values(ascending=False))
total_rev = rev_share.sum()
for tier, rev in rev_share.items():
    print(f"  {tier:<15} ${rev:>12,.0f}  ({rev/total_rev*100:.1f}%)")

print("\n[OK] Step 1 Complete — customers_enriched.csv ready for SQL + Power BI")
print("=" * 65)


STEP 1D — Exporting Enriched Dataset
  Saved: C:\Users\91888\OneDrive\Desktop\sqlcna_project\data\processed\customers_enriched.csv
  Shape: 3,900 rows × 31 columns

── ENGINEERED FEATURE SUMMARY ───────────────────
       promo_dependency_score  value_score  satisfaction_flag  estimated_annual_revenue  is_loyal_behavioral  is_loyal_value_sat
count                 3900.00      3900.00            3900.00                   3900.00              3900.00             3900.00
mean                    43.00        45.35               0.42                   1038.93                 0.16                0.21
std                     49.51        17.75               0.49                   1145.03                 0.37                0.41
min                      0.00         1.91               0.00                     20.00                 0.00                0.00
25%                      0.00        32.97               0.00                    188.00                 0.00                0.00
50%       

In [19]:
# Re-load the exported file to confirm it round-trips cleanly
df2 = pd.read_csv(OUT_DIR / "customers_enriched.csv")
print(f"  Reloaded: {df2.shape[0]:,} rows × {df2.shape[1]} columns")

  Reloaded: 3,900 rows × 31 columns